# Task 15 (server) — reproduce concept training before extending it

Everything lives under the directory this notebook runs from: clones in
`concept_aware/`, results in `outputs/`.  Set `GPU_ID` in the setup cell to a
free GPU before running.

In [ ]:
# Install once per environment.  transformers is pinned LAST and BELOW 4.58 on
# purpose: upstream's extractor reuses a prefix KV cache through
# DynamicCache.from_legacy_cache, which transformers removed in v5.
%pip install -q accelerate peft bitsandbytes datasets spacy "mteb>=1.12" nltk scipy scikit-learn seaborn pandas pytest wandb
%pip install -q "transformers>=4.51,<4.58"
# peft raises on torchao < 0.16 from inside PeftModel.from_pretrained, which is
# how every evaluator loads an adapter.
%pip install -q -U "torchao>=0.16"
!python -m spacy download en_core_web_sm
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

In [ ]:
import os
# Pick a free GPU BEFORE torch initialises CUDA.  Override without editing this
# file:  GPU_ID=1 jupyter nbconvert --execute ...
GPU_ID = os.environ.get("GPU_ID", "0")
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

from pathlib import Path
import hashlib, json, re, shutil, subprocess, sys, torch

# Every clone, dataset, checkpoint and result lives under BASE, so the whole
# experiment is one directory to archive or copy off the server.
BASE = Path(os.environ.get("CONCEPT_BASE", Path.cwd())).resolve()
WORK = BASE / "concept_aware"
MAIN = WORK / "concept-aware-training"     # our repo: patch, evaluators, scripts
EXT = WORK / "learning-concepts"           # upstream, pinned
DATA = WORK / "data"                       # CONCEPT_DATA_ROOT
RUNS = WORK / "runs"                       # adapters (small at r=4, kept)
OUTPUTS = BASE / "outputs"                 # everything you download for analysis

BASE_MODEL = "meta-llama/Llama-3.2-1B"
PRIMARY_SEED = 42
# One seed screens the pipeline and shows the direction of every effect, but it
# CANNOT support a claim: the pre-registered rule needs all three to agree in
# sign.  Flip to True for the reportable run; finished arms are skipped.
RUN_MULTISEED = False
SEEDS = [PRIMARY_SEED] + ([123, 2024] if RUN_MULTISEED else [])
UPSTREAM_COMMIT = "b1d414143d11c8ed988b4cccbb06626cc8272bbe"

# Capture an existing `huggingface-cli login` BEFORE redirecting HF_HOME.  The
# token lives under the DEFAULT HF_HOME, so once we move HF_HOME into the project
# directory a freshly spawned child finds no token and gated downloads 401 --
# even though the parent, which imported huggingface_hub earlier, looks fine.
# Exporting HF_TOKEN makes auth explicit and inherited by every subprocess.
_cli_token = Path.home() / ".cache" / "huggingface" / "token"
if not os.environ.get("HF_TOKEN") and _cli_token.is_file():
    os.environ["HF_TOKEN"] = _cli_token.read_text().strip()
os.environ["HF_HOME"] = str(WORK / "hf_cache")

RUN_DATA = True
RUN_SMOKE = True
RUN_SCREEN = True
RUN_CONFIRM = False
RUN_EVAL = True
# Skip any run whose artefacts already exist.  A killed job resumes from here.
RESUME_FINISHED_RUNS = True

# Extraction precision.  Upstream inherits use_4bit=True from TrainingConfig,
# where it exists for QLoRA TRAINING.  Extraction runs ~94 small forwards per
# sequence; measured on an L4, bf16 was ~25% faster and avoids quantisation
# noise in the top-100 pool and the 0.75 cosine threshold the method depends on.
EXTRACT_4BIT = False
# Full paper spec: 10,000 sequences split 8000/1000/1000.  merge_synonym_parts
# hard-fails unless the split sizes sum to the rows the shards actually cover.
EXTRACT_SEQUENCES = 10000
SPLIT_TRAIN = int(EXTRACT_SEQUENCES * 0.8)
SPLIT_VAL = SPLIT_TEST = int(EXTRACT_SEQUENCES * 0.1)

_BAR = re.compile(r"\b(\d+)/(\d+)\s*\[")   # tqdm counter, e.g. "  200/1000 ["
PROGRESS_EVERY = 100

def run(argv, cwd=None, env=None):
    """Run a child process, streaming its output and keeping the tail on failure."""
    argv = list(map(str, argv))
    print("+", " ".join(argv), flush=True)
    merged = os.environ.copy()
    merged.update({"CONCEPT_DATA_ROOT": str(DATA),
                   "CONCEPT_CHECKPOINT_ROOT": str(RUNS),
                   "CONCEPT_RESULTS_ROOT": str(OUTPUTS)})
    if env: merged.update(env)
    process = subprocess.Popen(argv, cwd=cwd, env=merged, text=True, bufsize=1,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in process.stdout:
        line = line.replace("\r", "")
        tail.append(line)
        del tail[:-40]
        hit = _BAR.search(line)
        if hit:
            done, total = int(hit.group(1)), int(hit.group(2))
            if done % PROGRESS_EVERY and done != total:
                continue
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(
            f"command failed with exit code {code}\n  {' '.join(argv)}\n"
            f"--- last {len(tail)} lines of its output ---\n{''.join(tail)}")

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def sync_small_artifacts(source, label):
    destination = OUTPUTS / label
    destination.mkdir(parents=True, exist_ok=True)
    for path in Path(source).rglob("*"):
        if path.is_file() and path.suffix.lower() in {".json", ".jsonl", ".csv", ".png", ".log"}:
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

def audit_outputs():
    """Keep OUTPUTS downloadable: reports only, no multi-GB weights."""
    forbidden = {"pytorch_model.bin", "model.safetensors", "optimizer.pt",
                 "scheduler.pt", "scaler.pt", "rng_state.pth"}
    found = [str(p) for p in OUTPUTS.rglob("*")
             if p.name in forbidden or p.name.startswith("checkpoint-")]
    assert not found, f"full-weight/optimizer artifacts reached OUTPUTS: {found}"

# The server filesystem persists, so the Colab Drive round-trip is unnecessary.
# These keep the same names and call sites as the Colab notebook, which is what
# lets the two share every experiment cell below.
def cache_dataset(leaf): pass

def restore_dataset(leaf):
    return (Path(leaf) / "synonyms_train.jsonl").is_file()

def cache_adapter(path): return Path(path)

def restore_adapter(path):
    return (Path(path) / "adapter_config.json").is_file()

RUN_MANIFEST = OUTPUTS / "run_manifests"

def save_runs(runs, name):
    RUN_MANIFEST.mkdir(parents=True, exist_ok=True)
    (RUN_MANIFEST / f"{name}.json").write_text(
        json.dumps({k: str(v) for k, v in runs.items()}, indent=2))

def load_runs(name):
    path = RUN_MANIFEST / f"{name}.json"
    return {} if not path.is_file() else {k: Path(v) for k, v in json.loads(path.read_text()).items()}

def restore_all(runs):
    live = {}
    for label, path in runs.items():
        if restore_adapter(path):
            live[label] = Path(path)
        else:
            print("missing adapter, dropping from this pass:", label)
    return live

def eval_done(marker):
    return Path(marker).is_file() and RESUME_FINISHED_RUNS

def assert_under_base(path):
    resolved = Path(path).resolve()
    assert str(resolved).startswith(str(BASE)), f"{resolved} escapes {BASE}"

for directory in (WORK, DATA, RUNS, OUTPUTS):
    directory.mkdir(parents=True, exist_ok=True)
print("BASE    ", BASE)
print("OUTPUTS ", OUTPUTS)
print("GPU     ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# Both clones are required: MAIN carries the patch, the evaluators and the
# summary script; EXT is the pinned upstream the patch applies to.
if not MAIN.exists():
    run(["git", "clone", "https://github.com/SharvaGogawale1/concept-aware-training.git", MAIN])
else:
    run(["git", "-C", str(MAIN), "pull", "--ff-only"])
if not EXT.exists():
    run(["git", "clone", "https://github.com/christine-zhang1/learning-concepts.git", EXT])
# Reset to the pinned commit and wipe every patch artefact before re-applying.
# Testing "does it apply, else does it reverse-apply" only worked while the patch
# never changed: once MAIN pulls a newer one, the old patch is applied, neither
# direction matches, and the run dies on an assertion.  Resetting is idempotent
# and always ends in the same state.  EXT holds upstream code only -- the corpus
# lives in DATA, outside it -- so clean -fd is safe.
run(["git", "-C", str(EXT), "reset", "--hard", UPSTREAM_COMMIT])
run(["git", "-C", str(EXT), "clean", "-fdq"])

patch_file = MAIN / "external" / "learning-concepts.patch"
assert patch_file.exists(), f"{patch_file} missing; push it before running here."
run(["git", "apply", str(patch_file)], cwd=EXT)
print("patch applied onto", UPSTREAM_COMMIT[:7])

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(EXT), "--no-deps"])

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

run([sys.executable, MAIN / "builddataset/verify_task14_data.py",
     "--repo_root", MAIN, "--download_missing",
     "--report_json", OUTPUTS / "external_benchmark_integrity.json"], cwd=MAIN)
(OUTPUTS / "environment_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))

# Llama-3.2-1B is gated.  Fail here rather than 20 minutes later inside a child
# process: a missing token surfaces as a 401 on config.json from a subprocess
# whose traceback says nothing about authentication.
from huggingface_hub import login, whoami
assert os.environ.get("HF_TOKEN"), (
    "No Hugging Face token.  Run `huggingface-cli login` on this machine, or "
    "export HF_TOKEN before starting Jupyter, then re-run this cell.")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hugging Face:", whoami()["name"])
subprocess.run([sys.executable, "-c",
                "from transformers import AutoConfig;"
                "AutoConfig.from_pretrained('meta-llama/Llama-3.2-1B');"
                "print('gated repo reachable from a subprocess')"],
               env={**os.environ}, check=True)

## Rebuild and audit the exact 8k/1k/1k data

The audit hard-fails on split overlap, target misalignment, empty sets, or any concept that is not a complete single token. The observed target is part of every set by construction.


In [ ]:
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
if RUN_DATA:
    # The extraction is the longest stage.  Each 1k shard is cached to Drive once
    # it completes, so a disconnect costs at most the shard in progress.
    restore_dataset(LEAF)
    # get_content_words.py streams into combined.jsonl, so an interrupted pass
    # leaves a SHORT file behind and existence alone does not mean completion.
    # Test for "enough rows", not "exactly EXTRACT_SEQUENCES": MAX_SAMPLES is
    # hardcoded to 10000 in that script, so the file legitimately holds 10000 rows
    # even when we only extract the first 4000, and an equality test would delete
    # and regenerate it on every resume.
    combined = LEAF.parent / "combined.jsonl"
    if not combined.is_file() and REUSE_CONTENT_WORDS_FROM:
        from transformers import AutoTokenizer
        assert (AutoTokenizer.from_pretrained(BASE_MODEL).get_vocab()
                == AutoTokenizer.from_pretrained(REUSE_CONTENT_WORDS_FROM).get_vocab()), (
            f"{BASE_MODEL} and {REUSE_CONTENT_WORDS_FROM} do not share a vocabulary, "
            "so their content words differ and must be regenerated")
        donor_tag = REUSE_CONTENT_WORDS_FROM.split("/")[-1].lower()
        donor = DATA / "c4" / donor_tag / "combined.jsonl"
        if not donor.is_file():
            restore_dataset(DATA / "c4" / donor_tag / "embedding")
        combined.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(donor, combined)
        print("reused content words from", REUSE_CONTENT_WORDS_FROM)
    if combined.is_file():
        rows = sum(1 for _ in combined.open())
        if rows < EXTRACT_SEQUENCES:
            print(f"combined.jsonl has {rows} rows, need {EXTRACT_SEQUENCES}; regenerating")
            combined.unlink()
        else:
            print(f"combined.jsonl has {rows} rows, using the first {EXTRACT_SEQUENCES}")
    if not combined.is_file():
        run([sys.executable, "data/get_content_words.py", "--model", BASE_MODEL,
             "--dataset", "c4", "--max_length", "256"], cwd=EXT)
        cache_dataset(LEAF)
    # An interrupted shard leaves PARTIAL synonyms_/topk_ files behind, so their
    # mere existence does not mean the shard finished.  Record completion in a
    # manifest on disk instead; embedding_synonyms.py truncates both files when
    # it restarts a shard, so a re-run is always clean.
    # One manifest per model: a shared name would let the 1B shards mark the 3B
    # ones as finished, and extraction would be skipped entirely.
    shards_done = load_runs(f"task15_shards{_TAG_SUFFIX}")
    for start in range(0, EXTRACT_SEQUENCES, 1000):
        end = start + 1000
        key = f"{start}_{end}"
        synonym_part = LEAF / f"synonyms_{start}_{end}.jsonl"
        topk_part = LEAF.parent / "prompting" / f"topk_{start}_{end}.jsonl"
        if (RESUME_FINISHED_RUNS and key in shards_done
                and synonym_part.is_file() and topk_part.is_file()):
            print("resume: extraction shard already complete", start, end)
            continue
        run([sys.executable, "data/embedding_synonyms.py", "c4",
             "--start", start, "--end", end, "--model", BASE_MODEL,
             *([] if EXTRACT_4BIT else ["--no-4bit"])], cwd=EXT)
        cache_dataset(LEAF)
        shards_done[key] = synonym_part
        save_runs(shards_done, f"task15_shards{_TAG_SUFFIX}")
    run([sys.executable, "data/merge_synonym_parts.py", "--train-size", SPLIT_TRAIN,
         "--val-size", SPLIT_VAL, "--test-size", SPLIT_TEST, "--expected-count", "2", "--force"], cwd=EXT)
    run([sys.executable, "data/augment_synonyms.py", "--base-dir", DATA,
         "--num-augmentations", "4", "--seed", "42", "--overwrite"], cwd=EXT)
    run([sys.executable, "data/randomize_synonyms.py", "--split", "train", "--overwrite"], cwd=EXT)

if RUN_DATA:
    cache_dataset(LEAF)
if RUN_DATA:
    run([sys.executable, "data/audit_concept_data.py",
         "--train", LEAF / "synonyms_train.jsonl",
         "--validation", LEAF / "synonyms_val.jsonl",
         "--test", LEAF / "synonyms_test.jsonl",
         "--tokenizer", BASE_MODEL,
         "--expected-train", SPLIT_TRAIN, "--expected-validation", SPLIT_VAL,
         "--expected-test", SPLIT_TEST,
         "--report", OUTPUTS / "data_audit.json"], cwd=EXT)

## Unit tests and eight-row GPU smoke run

The smoke run checks the complete QLoRA path before any sweep. It is deleted immediately. The tests cover the released loss, our optional objectives, gradients, and hierarchy sequence scoring.


In [ ]:
if RUN_SMOKE:
    run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=EXT)
    # No map-style preprocessing cache is used. Two independent loads must
    # still produce identical candidate supervision.
    # pip install -e EXT only installs the conceptlib PACKAGE (all pyproject
    # declares); train.py is a loose top-level module, so importing it in-process
    # needs EXT on sys.path.  The subprocess calls are unaffected -- they pass
    # cwd=EXT -- which is why this only bites the in-notebook import.
    if str(EXT) not in sys.path:
        sys.path.insert(0, str(EXT))
    from train import ConceptDataset
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    first = ConceptDataset(LEAF / "synonyms_train.jsonl", tokenizer, max_samples=8)
    second = ConceptDataset(LEAF / "synonyms_train.jsonl", tokenizer, max_samples=8)
    assert [x["content_words"] for x in first.data] == [x["content_words"] for x in second.data]
    smoke = RUNS / "smoke"
    assert_under_base(smoke)
    run([sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
         "--dataset-type", "embedding", "--concept-loss-weight", "1.0",
         "--max-train-samples", "8", "--num-train-epochs", "1",
         "--output-dir", smoke, "--save-strategy", "no", "--report-to", "none"], cwd=EXT)
    assert (smoke / "adapter_config.json").exists()
    # Required one-model equivalence test: adapter logits and merged logits.
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    adapted = PeftModel.from_pretrained(base, smoke).eval()
    probe = tokenizer("A dog is an animal.", return_tensors="pt")
    with torch.no_grad(): adapter_logits = adapted(**probe).logits
    merged = adapted.merge_and_unload().eval()
    with torch.no_grad(): merged_logits = merged(**probe).logits
    assert torch.allclose(adapter_logits, merged_logits, atol=2e-4, rtol=2e-4)
    del merged, adapted, base
    shutil.rmtree(smoke)

## Exact reproduction schedule

Seed 42 gets the complete $\lambda\in\{0.25,0.5,0.75,1\}$ curve. The headline NTP, one-epoch augmented NTP, randomized $\lambda=.25$, and concept-marginal $\lambda=1$ settings are then confirmed with seeds 42, 123, and 2024. Released effective batch size is logged. If the headline fails, only seed 42 is rerun with the paper-stated batch before any diagnosis.


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
# The server filesystem persists, so this is a cheap existence check.
restore_dataset(LEAF)

def adapter_path(method, seed, value):
    path = RUNS / MODEL_TAG / method / f"seed_{seed}" / str(value)
    assert_under_base(path)
    return path

def finished(path):
    """A run counts as finished when its adapter exists locally or on Drive."""
    if (Path(path) / "adapter_config.json").is_file():
        return True
    return restore_adapter(path)

def train_flat(method, seed, concept_weight, *, objective="set_marginal",
               slot_ntp_weight=None, contrast_beta=0.0,
               randomized=False, data_augmentation=False, epochs=5, train_file=None,
               max_samples=None, batch=8, accum=2):
    out = adapter_path(method, seed, f"lambda_{concept_weight}_beta_{contrast_beta}")
    args = [sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
            "--dataset-type", "embedding", "--concept-loss-weight", concept_weight,
            "--concept-objective", objective, "--contrast-beta", contrast_beta,
            "--seed", seed, "--num-train-epochs", epochs, "--output-dir", out,
            "--save-strategy", "no", "--report-to", "none",
            "--per-device-train-batch-size", batch,
            "--gradient-accumulation-steps", accum]
    if slot_ntp_weight is not None: args += ["--slot-ntp-weight", slot_ntp_weight]
    if randomized: args += ["--randomized-synonyms"]
    if data_augmentation: args += ["--use-data-augmentation"]
    if train_file: args += ["--train-file", train_file]
    if max_samples: args += ["--max-train-samples", max_samples]
    # Released effective batch is 8 x 2 = 16; it is recorded in every config.
    if RESUME_FINISHED_RUNS and finished(out):
        print("resume: already trained, skipping", out)
        return out
    run(args, cwd=EXT)
    cache_adapter(out)
    return out

In [ ]:
REPRO_RUNS = load_runs(f"task15{_TAG_SUFFIX}")
if RUN_SCREEN:
    REPRO_RUNS["ntp_seed42"] = train_flat("ntp", 42, 0.0)
    for lam in [0.25, 0.5, 0.75, 1.0]:
        REPRO_RUNS[f"zhang_lambda{lam}_seed42"] = train_flat("zhang_marginal", 42, lam)
    REPRO_RUNS["augmented_ntp_seed42"] = train_flat("augmented_ntp", 42, 0.0, epochs=1,
        train_file=LEAF / "synonyms_train_aug5x.jsonl", data_augmentation=True)
    REPRO_RUNS["randomized_seed42"] = train_flat("randomized", 42, 0.25, randomized=True)

if RUN_CONFIRM:
    for seed in SEEDS:
        REPRO_RUNS[f"ntp_seed{seed}"] = train_flat("ntp", seed, 0.0)
        REPRO_RUNS[f"augmented_ntp_seed{seed}"] = train_flat("augmented_ntp", seed, 0.0,
            epochs=1, train_file=LEAF / "synonyms_train_aug5x.jsonl", data_augmentation=True)
        REPRO_RUNS[f"randomized_seed{seed}"] = train_flat("randomized", seed, 0.25, randomized=True)
        REPRO_RUNS[f"zhang_seed{seed}"] = train_flat("zhang_marginal", seed, 1.0)
    # The lambda sweep already trained zhang_marginal at lambda=1.0 on seed 42, and
    # adapter_path() maps both calls to the same directory.  Two dict keys pointing at
    # one adapter would score it twice and report it as two arms.
    REPRO_RUNS.pop("zhang_lambda1.0_seed42", None)
save_runs(REPRO_RUNS, f"task15{_TAG_SUFFIX}")

# Use only if the released-batch seed-42 reproduction misses the stated trend.
# This is a named sensitivity run, never a replacement or a cherry-picked row.
RERUN_PAPER_STATED_BATCH = False
if RERUN_PAPER_STATED_BATCH:
    REPRO_RUNS["zhang_seed42_paper_batch"] = train_flat(
        "zhang_marginal_paper_batch", 42, 1.0, batch=2, accum=1)

## Evaluation and learning curves

Every checkpoint is scored by the same evaluators. “Global NLL” covers every next-token position; “content-word NLL” covers the semantic slots; “set mass” is total probability assigned to the gold-inclusive valid set. SWORDS and bm-semlex are zero-shot here.


In [ ]:
if RUN_EVAL:
    # A fresh session has the manifest but not the weights; pull them back first.
    REPRO_RUNS = restore_all(REPRO_RUNS)
    checkpoints = [BASE_MODEL, *map(str, REPRO_RUNS.values())]
    result_dir = OUTPUTS / RESULT_DIR
    result_dir.mkdir(parents=True, exist_ok=True)
    # Each evaluator is skipped only when its own output already scores every
    # checkpoint of this pass, so a disconnect costs at most one evaluator
    # instead of the whole section.
    guarded(result_dir / "perplexity.json", checkpoints,
            [sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "perplexity.json"], EXT, "perplexity")
    guarded(result_dir / "concept_sets.json", checkpoints,
            [sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "concept_sets.json"], EXT, "concept sets")
    for label, checkpoint in {"pretrained": BASE_MODEL, **REPRO_RUNS}.items():
        display_label = label.replace("_", " ")
        csv_output = result_dir / f"sts_{display_label}.csv"
        # STS writes one CSV per checkpoint, so it resumes per checkpoint.
        if sts_covered(csv_output):
            print("resume: STS already scored, skipping", display_label)
            continue
        mteb_args = [sys.executable, "eval/eval_mteb.py", "--base-model", BASE_MODEL,
                     "--dataset", "c4", "--dataset-type", "embedding", "--tasks", "sts",
                     "--run-label", display_label,
                     "--csv-output", csv_output,
                     "--mteb-output-root", result_dir / "mteb_raw"]
        mteb_args += ["--no-adapter"] if checkpoint == BASE_MODEL else ["--adapter-path", checkpoint]
        run(mteb_args, cwd=EXT)
    guarded(result_dir / "swords_zero_shot.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--swords_json", MAIN / "data/swords/swords-v1.1_dev.json.gz",
             "--results_json", result_dir / "swords_zero_shot.json", "--modes", "left", "full"],
            MAIN, "SWORDS")
    # Seconds to run, and each must always match the SWORDS file it reads, so
    # these are never skipped.  Pretrained is the reference row; NTP and
    # augmented NTP are the matched controls the causal claims are stated
    # against.  The index is looked up by label rather than hardcoded: RUN_CONFIRM
    # pops zhang_lambda1.0_seed42, so positions shift once seeds are added and a
    # literal index would quietly compare against the wrong arm.
    # randomized is the one that decides whether the SWORDS gain is semantic:
    # it is trained with the same objective on meaningless candidate sets, so a
    # concept arm that does not separate from it there is buying its GAP with
    # distributional smoothing rather than with concept content.
    baselines = {"pretrained": BASE_MODEL}
    for label in ("ntp_seed42", "augmented_ntp_seed42", "randomized_seed42"):
        if label in REPRO_RUNS:
            baselines[label] = str(REPRO_RUNS[label])
    for label, reference in baselines.items():
        run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
             "--kind", "swords", "--results-json", result_dir / "swords_zero_shot.json",
             "--baseline-index", checkpoints.index(reference),
             "--output", result_dir / f"swords_paired_ci_vs_{label}.json"], cwd=MAIN)
    guarded(result_dir / "bm_semlex.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_bm_semlex.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--data", MAIN / "data/bm_semlex/curated_200.tsv",
             "--results_json", result_dir / "bm_semlex.json"], MAIN, "bm-semlex")
    manifest = {"pretrained": BASE_MODEL, **{label.replace("_", " "): str(path) for label, path in REPRO_RUNS.items()}}
    (result_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    run([sys.executable, MAIN / "scripts/summarize_concept_experiments.py",
         "--manifest", result_dir / "manifest.json", "--result-dir", result_dir,
         "--output", result_dir / "flat_main_table.csv"], cwd=MAIN)
    for label, path in REPRO_RUNS.items(): sync_small_artifacts(path, f"{LOG_DIR}/{label}")
    audit_outputs()

In [ ]:
# Plot only scalar learning curves; adapters stay under RUNS.
import pandas as pd
from matplotlib import pyplot as plt
curves = []
for label, path in REPRO_RUNS.items():
    history = Path(path) / "training_history.jsonl"
    if history.exists():
        frame = pd.read_json(history, lines=True)
        frame["method"] = label
        curves.append(frame)
if not curves:
    print("no training_history.jsonl found; nothing to plot")
else:
    frame = pd.concat(curves, ignore_index=True)
    # The trainer logs ce_loss and concept_loss once per EVAL BATCH, two or three
    # rows sharing one global_step.  Only the row that also carries eval_loss is
    # the aggregate over the validation set; plotting the rest draws intra-eval
    # scatter as if it were training dynamics.
    train_rows = frame.dropna(subset=["loss"])
    eval_rows = frame.dropna(subset=["eval_loss"])
    panels = [(train_rows, "step", "loss", "training loss"),
              (eval_rows, "epoch", "ce_loss", "validation NTP cross-entropy"),
              (eval_rows, "epoch", "concept_loss", "validation concept loss")]
    figure, axes = plt.subplots(1, 3, figsize=(12, 3.4))
    for axis, (rows, x, y, title) in zip(axes, panels):
        if y not in rows:
            continue
        for label, group in rows.groupby("method"):
            group = group.dropna(subset=[y])
            if not group.empty and group[y].abs().sum() > 0:
                axis.plot(group[x], group[y], marker="o", ms=3, lw=1.4, label=label)
        axis.set_xlabel(x); axis.set_title(title, fontsize=10); axis.grid(alpha=.25, lw=.5)
    axes[0].legend(fontsize=7, frameon=False, ncol=2)
    plt.tight_layout()
    plt.savefig(OUTPUTS / RESULT_DIR / "training_curves.png", dpi=180)
    plt.show()

## Reproduction gate

Proceed only if Zhang concept marginal beats NTP and augmented NTP on mean STS, improves content-word perplexity over NTP, stays near pretrained global perplexity, and the $\lambda$ curve has the reported direction. If seed 42 fails, rerun that one setting with the paper-stated effective batch and report both configurations—do not silently substitute it.
